### Data Ingestion 
Before we can ask legal questions, the AI needs a database of legal knowledge

#### Goal
- understand how raw text is convert into a format AI can understand
- loading pdf files using langchain
- text splitting
- vectorization

In [1]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyMuPDFLoader # used to load pdf documents into langchain format
import pymupdf # underlying library for the previous module
import pandas as pd
from IPython.display import display, HTML

load_dotenv()

PERSIST_DIRECTORY = "./chroma_db"

In [2]:
file_path = os.path.join(r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data\Indian_Constitution.pdf")

In [3]:
# splitting the document pagewise
loader = PyMuPDFLoader(file_path)
data = loader.load()
print(data[0])

page_content='£ÉÉ®iÉ BÉEÉ ºÉÆÉÊ´ÉvÉÉxÉ 
[1
, 2024
]
THE CONSTITUTION OF INDIA
[As on 1st May, 2024] 
2024
GOVERNMENT OF INDIA
MINISTRY OF LAW AND JUSTICE
LEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING' metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 0}


We index preamble seperatly

In [4]:
# articles short
#Pages 3 to 31 contains the  introduction
articles_short = data[3:31]
articles_short[0]

Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 3}, page_content='THE CONSTITUTION OF INDIA\n____________                                                                    \nCONTENTS\n__________\n                                                                                          \nPREAMBLE\nPART I \nTHE UNION AND ITS TERRITORY\nARTICLES \n1.\nName and territory of the Union.\n  2. \nAdmission or establishment of new States. \n[2A.         Sikkim to be associated with the Union.—Omitted.]\n3.\nFormation of new States and alteration of areas, boundar

In [5]:
# preamble
preamble = data[31]
preamble

Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 31}, page_content='THE CONSTITUTION OF INDIA \nPREAMBLE\nWE, THE PEOPLE OF INDIA, having solemnly resolved to constitute \nIndia into a \n1[SOVEREIGN SOCIALIST SECULAR DEMOCRATIC \nREPUBLIC] and to secure to all its citizens:\nJUSTICE, social, economic and political;\n \nLIBERTY of thought, expression, belief, faith and worship;\nEQUALITY of status and of opportunity;\nand to promote among them all\nFRATERNITY assuring the dignity of the individual and the 2[unity \nand integrity of the Nation];\nIN OUR CONS

In [6]:
articles = data[32:283]

In [7]:
print(articles[0].page_content)

2
PART I
THE UNION AND ITS TERRITORY
1. Name and territory of the Union.—(1) India, that is Bharat, 
shall be a Union of States.
1[(2) The States and the territories thereof shall be as specified in 
the First Schedule.]
(3) The territory of India shall comprise—
(a) the territories of the States; 
2[(b) the Union territories specified in the First Schedule; 
and]
(c) such other territories as may be acquired.
2. Admission or establishment of new States.—Parliament may 
by law admit into the Union, or establish, new States on such terms and 
conditions as it thinks fit.
3[2A. [Sikkim to be associated with the Union.] —Omitted by the 
Constitution (Thirty-sixth Amendment) Act, 1975, s. 5 (w.e.f. 26-4-1975).]
3. Formation of new States and alteration of areas, 
boundaries or names of existing States.—Parliament may by law—
(a) form a new State by separation of territory from any 
State or by uniting two or more States or parts of States or by 
uniting any territory to a part of any State

In [8]:

##code to remove headers  and footers from pdf
import re
for art in articles:
  art.page_content = re.split(r'_{2,}', art.page_content)[0].strip()

we split the article based on parts since constitution is divided into 25 parts

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

separator = "PART "  # Common chapter heading pattern

# Initialize the RecursiveCharacterTextSplitter with your chapter separator
text_splitter = RecursiveCharacterTextSplitter(
    separators=[separator],  # List of separators used to split the document
    chunk_size=1000,         # Maximum size of each split chunk
    chunk_overlap=100          # No overlap between splits
)

# Concatenate the 'page_content' of all Document objects into a single string
text = "".join([doc.page_content for doc in articles])

# Split the combined text by chapters
chapters = text_splitter.split_text(text)
chapters = chapters[1:]

In [10]:
chapters[7]

'PART VII\n[The States in Part B of the First Schedule].112'

In [11]:
def split_article_wise(chapter):
  """ Split the text based on Articles"""
  sections = re.split(r'(?=\n\d+\[?\d*[a-zA-Z]?[a-zA-Z]?\.\s)', chapter)
  # Optional: Strip leading/trailing whitespaces from each section
  sections = [section.strip() for section in sections]
  return sections

In [12]:
articles=[]
for chapter in chapters:
  articles.append(split_article_wise(chapter))

In [13]:
def cleaning_articles(arti):
  """Cleaning the articles text for unwanted special characters"""
  cl_art=[]
  for i in arti:
    pattern = r'^\d+\['
    if re.match(pattern, i):
      pos=arti.index(i)
      i = i.replace(i[:2], "", 1)
      arti[pos] = i
      cl_art.append(i)
    else:
      cl_art.append(i)
  return cl_art

In [14]:
docs = [cleaning_articles(i) for i in articles]

In [15]:
def adding_tag(docs):
  '''For adding Article keyword before each article number'''
  d=[docs[0]]
  for i in range(1,len(docs)):
    docs[i] = "Article " + docs[i]
    d.append(docs[i])
  return d

In [16]:
documents = [adding_tag(i) for i in docs]

In [17]:
nn =[i[1:] for i in documents]

In [18]:
nn[0][0].split(".")[0]

'Article 1'

In [19]:
nn[0]

['Article 1. Name and territory of the Union.—(1) India, that is Bharat, \nshall be a Union of States.\n1[(2) The States and the territories thereof shall be as specified in \nthe First Schedule.]\n(3) The territory of India shall comprise—\n(a) the territories of the States; \n2[(b) the Union territories specified in the First Schedule; \nand]\n(c) such other territories as may be acquired.',
 'Article 2. Admission or establishment of new States.—Parliament may \nby law admit into the Union, or establish, new States on such terms and \nconditions as it thinks fit.',
 'Article 2A. [Sikkim to be associated with the Union.] —Omitted by the \nConstitution (Thirty-sixth Amendment) Act, 1975, s. 5 (w.e.f. 26-4-1975).]',
 'Article 3. Formation of new States and alteration of areas, \nboundaries or names of existing States.—Parliament may by law—\n(a) form a new State by separation of territory from any \nState or by uniting two or more States or parts of States or by \nuniting any territory 

In [20]:
[j.split(".")[0] for i in nn for j in i]

['Article 1',
 'Article 2',
 'Article 2A',
 'Article 3',
 'Article 4',
 'Article 5',
 'Article 6',
 'Article 7',
 'Article 8',
 'Article 9',
 'Article 10',
 'Article 11',
 'Article 12',
 'Article 13',
 'Article 14',
 'Article 15',
 'Article 16',
 'Article 17',
 'Article 18',
 'Article 19',
 'Article 20',
 'Article 21',
 'Article 21A',
 'Article 22',
 'Article 23',
 'Article 24',
 'Article 25',
 'Article 26',
 'Article 27',
 'Article 28',
 'Article 29',
 'Article 30',
 'Article 31',
 'Article 31A',
 'Article 31B',
 'Article 31C',
 'Article 32',
 'Article 132A',
 'Article 33',
 'Article 34',
 'Article 35',
 'Article 36',
 'Article 37',
 'Article 38',
 'Article 39',
 'Article 39A',
 'Article 40',
 'Article 41',
 'Article 42',
 'Article 43',
 'Article 43A',
 'Article 43B',
 'Article 44',
 'Article 45',
 'Article 46',
 'Article 47',
 'Article 48',
 'Article 48A',
 'Article 49',
 'Article 50',
 'Article 51',
 'Article 51A',
 'Article 52',
 'Article 53',
 'Article 54',
 'Article 55',
 'Articl

In [21]:
const = [" ".join(i) for i in documents]

In [22]:
const[0].replace("\n"," ")

'PART I THE UNION AND ITS TERRITORY Article 1. Name and territory of the Union.—(1) India, that is Bharat,  shall be a Union of States. 1[(2) The States and the territories thereof shall be as specified in  the First Schedule.] (3) The territory of India shall comprise— (a) the territories of the States;  2[(b) the Union territories specified in the First Schedule;  and] (c) such other territories as may be acquired. Article 2. Admission or establishment of new States.—Parliament may  by law admit into the Union, or establish, new States on such terms and  conditions as it thinks fit. Article 2A. [Sikkim to be associated with the Union.] —Omitted by the  Constitution (Thirty-sixth Amendment) Act, 1975, s. 5 (w.e.f. 26-4-1975).] Article 3. Formation of new States and alteration of areas,  boundaries or names of existing States.—Parliament may by law— (a) form a new State by separation of territory from any  State or by uniting two or more States or parts of States or by  uniting any ter

In [23]:
# code to generate chapters names in order to add in the metadata

z2_list = []
for chapter in chapters:
    z2 = chapter.split('\n')[0]
    z2_list.append(z2)

z2_list

['PART I',
 'PART II',
 'PART III ',
 'PART IV',
 'PART  IVA',
 'PART  V',
 'PART VI',
 'PART VII',
 'PART VIII ',
 'PART IX',
 'PART IXA',
 'PART IXB ',
 'PART X',
 'PART XI',
 'PART XII',
 'PART XIII',
 'PART XIV',
 'PART  XIVA',
 'PART XV',
 'PART  XVI',
 'PART XVII',
 'PART XVIII',
 'PART XIX',
 'PART XX',
 'PART XXI',
 'PART XXII']

In [24]:
len(z2_list)

26

In [25]:
part_names = ['The union and its territories',
 'Citizenship',
 'Fundamental rights',
 'Directive principles of state policy',
 'Fundamental duties ',
 'The union',
 'The states',
 'The States in Part B of the First Schedule',
 'The union territories',
 'The panchayats',
 'The municipalities',
 'The co-operative societies',
 'The scheduled and tribal areas',
 'Relations between the union and the states',
 'Finance, property, contracts and suits',
 'Trade, commerce and intercourse within the territory of india',
 'Services under the union and the states',
 'Tribunals',
 'Elections',
 'Special provisions relating to certain classes',
 'Official language',
 'Emergency provisions',
 'Miscellaneous',
 'Amendment of the constitution',
 'Temporary, transitional and special provisions',
 'Short title, commencement,authoritative text in hindi and repeals']

In [26]:
len(part_names)

26

In [27]:
z1_list = [" ".join(j.capitalize() for j in i.split(" ")) for i in part_names]

In [28]:
z2_list = [i.strip() for i in z2_list]

In [29]:
z2_list,z1_list

(['PART I',
  'PART II',
  'PART III',
  'PART IV',
  'PART  IVA',
  'PART  V',
  'PART VI',
  'PART VII',
  'PART VIII',
  'PART IX',
  'PART IXA',
  'PART IXB',
  'PART X',
  'PART XI',
  'PART XII',
  'PART XIII',
  'PART XIV',
  'PART  XIVA',
  'PART XV',
  'PART  XVI',
  'PART XVII',
  'PART XVIII',
  'PART XIX',
  'PART XX',
  'PART XXI',
  'PART XXII'],
 ['The Union And Its Territories',
  'Citizenship',
  'Fundamental Rights',
  'Directive Principles Of State Policy',
  'Fundamental Duties ',
  'The Union',
  'The States',
  'The States In Part B Of The First Schedule',
  'The Union Territories',
  'The Panchayats',
  'The Municipalities',
  'The Co-operative Societies',
  'The Scheduled And Tribal Areas',
  'Relations Between The Union And The States',
  'Finance, Property, Contracts And Suits',
  'Trade, Commerce And Intercourse Within The Territory Of India',
  'Services Under The Union And The States',
  'Tribunals',
  'Elections',
  'Special Provisions Relating To Certain 

We use proper chunking of 1000 and overlap of 300

In [30]:
from langchain_text_splitters import CharacterTextSplitter
chunked_list = []
t_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=300,
    length_function=len,
    is_separator_regex=True,
)

In [31]:
for i,chapter in enumerate(const):
  # adding meta data here li is the list with name of chapters which we created above
  metadatas = [{"source": "https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf",
                "source_name": "Indian Constitution", "country":"India","db_owner":"LawGlance","part":z2_list[i],"part_name":z1_list[i]}]
  # Use the splitter to split each chapter into chunks
  chunks = t_splitter.create_documents([chapter],metadatas=metadatas)
  # Append the chunks to the chunked_list
  chunked_list.append(chunks)

In [32]:
chunked_list[1]

[Document(metadata={'source': 'https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf', 'source_name': 'Indian Constitution', 'country': 'India', 'db_owner': 'LawGlance', 'part': 'PART II', 'part_name': 'Citizenship'}, page_content='PART II\nCITIZENSHIP Article 5. Citizenship at the commencement of the Constitution.—At the \ncommencement of this Constitution, every person who has his domicile in the \nterritory of India and—\n(a) who was born in the territory of India; or \n(b) either of whose parents was born in the territory of India; or\n(c) who has been ordinarily resident in the territory of India for \nnot less than five years immediately preceding such commencement,  \nshall be a citizen of India. Article 6. Rights of citizenship of certain persons who have migrated to \nIndia from Pakistan.—Notwithstanding anything in article 5, a person who \nhas migrated to the territory of India from the territory now included in \nPakistan sha

In [ ]:
def flatten_extend(matrix):
    flat_list = []
    for row in matrix:
        flat_list.extend(row)
    return flat_list

we use huggingface embeddings in our production

In [35]:
# setup the llm
llm = ChatGroq(model= "llama-3.3-70b-versatile",
               temperature=0.9,
               api_key= os.getenv("GROQ_API_KEY") # Optional if the env var is named exactly GROQ_API_KEY
               ) 

embeddings =  HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


###  Vector Database Strategy: Distance Metrics
When initializing the Chroma vector store, the choice of "Space" determines how the AI calculates the similarity between a user's question and the legal documents.

####  Option A: Cosine Similarity (Recommended for LawGlance)
By adding `collection_metadata={"hnsw:space": "cosine"}`, we force the database to use **Cosine Similarity**.
* **The Math:** Scores range from `0.0` (no match) to `1.0` (perfect match).
* **Benefit:** This works natively with LangChain's `similarity_score_threshold`. A threshold of `0.3` means *"Only show results that are at least 30% relevant."*
* **Implementation:**
```python
vectorstore = Chroma.from_documents(
    documents=do,
    embedding=embeddings,
    persist_directory="chroma_db_legal_bot_part1",
    collection_metadata={"hnsw:space": "cosine"} # <--- Ensures 0 to 1 scoring
)

In [36]:
# class method used to initialize the database for the first time
# by providing the raw data
"""
What it does: It takes your list of Document objects, runs them through the embedding model to turn text into numbers (vectors), and then saves them into the specified persist_directory.

When to use it: Use this during the Ingestion Phase. You run this once to "build" your knowledge base.

Analogy: This is like buying a bookshelf and physically putting books on it for the first time.
"""

vectorstore = Chroma.from_documents(
    documents=do,
    embedding= embeddings,
    persist_directory="chroma_db_legal_bot_part1",
    collection_metadata={"hnsw:space": "cosine"} # <--- Crucial line
)


# vector_store.add_documents(documents=do)

In [37]:
# loads the vector store

"""
This is the Class Constructor used to connect to an existing database that has already been saved to your disk.

What it does: It looks at the persist_directory on your computer. If it finds an existing Chroma database there, it links it to your code using the embedding_function so it knows how to "read" the stored vectors. It does not add new documents; it just prepares the database for searching.

When to use it: Use this in your Main Application (lawglance_main.py or app.py). You don't want to re-process the PDF every time a user asks a question; you just want to open the existing database.

Analogy: This is like walking up to a bookshelf that is already full and opening it so you can look for information.
"""

vector_store = Chroma(
    persist_directory="chroma_db_legal_bot_part1",
    embedding_function=embeddings
)

retriever = vector_store.as_retriever(
    search_type = "similarity", 
    search_kwargs={"k": 10}
)

retriever.invoke("What is article 1")

[Document(id='5f9e84ef-048c-4385-8b69-2ddfc8701067', metadata={'db_owner': 'LawGlance', 'source': 'https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf', 'country': 'India', 'source_name': 'Indian Constitution', 'part': 'PART XVIII', 'part_name': 'Emergency Provisions'}, page_content='containing such a recital.] Article 359. Suspension of the enforcement of the rights conferred by \nPart III during emergencies.—(1) Where a Proclamation of Emergency is in \noperation, the President may by order declare that the right to move any court \nfor the enforcement of such of 1[the rights conferred by Part III (except articles \n20 and 21)] as may be mentioned in the order and all proceedings pending in \nany court for the enforcement of the rights so mentioned shall remain \nsuspended for the period during which the Proclamation is in force or for such \nshorter period as may be specified in the order.\n2[(1A) While an order made under clause (1

In [38]:
def footer_remover(dat):
  for art in dat:
    art.page_content = re.split(r'_{2,}', art.page_content)[0].strip()
  return(" ".join([i.page_content for i in dat]))

In [39]:
first_schedule = footer_remover(data[283:294])

In [40]:
second_schedule = footer_remover(data[293:300])
print(second_schedule)

263
SECOND SCHEDULE
[Articles 59(3), 65(3), 75(6), 97, 125, 148(3), 158(3), 164 (5), 186 and 221] 
PART A 
PROVISIONS AS TO THE PRESIDENT AND THE GOVERNORS OF STATES 1***
1. There shall be paid to the President and to the Governors of the States
1***  the following emoluments per mensem, that is to say:—
The President
……                10,000 rupees .
The Governor of a State     ……   
5,500 rupees .
2. There shall also be paid to the President and to the Governors of the 
States 2***  such allowances as were payable respectively to the Governor-
General of the Dominion of India and to the Governors of the corresponding 
Provinces immediately before the commencement of this Constitution.
3. The President and the Governors of 3[the States] throughout their respective 
terms of office shall be entitled to the same privileges to which the Governor-
General and the Governors of the corresponding Provinces were respectively 
entitled immediately before the commencement of this Constitution.


In [41]:
third_schedule = footer_remover(data[299:303])
third_schedule

'269\nTHIRD SCHEDULE\n[Articles 75(4), 99, 124(6), 148(2), 164(3), 188 and 219]   \nForms  of  Oaths  or Affirmations \nI\nForm of oath of office for a Minister for the Union:—\n“I,   A. B.,  do  swear   in   the  name   of   God  that I will bear true faith\n                                  solemnly affirm   \nand allegiance to  the  Constitution of India as by law established, 1[that I \nwill uphold the sovereignty and integrity of India,] that I will faithfully \nand conscientiously discharge my duties as a Minister for the Union and \nthat I will do right to all manner of people in accordance with the \nConstitution and the law, without fear or favour, affection or ill-will.”\nII\nForm of oath of secrecy for a Minister for the Union:—\n“I,   A.B.,  do swear  in  the  name  of  God that  I  will  not  directly  or\nsolemnly affirm\nindirectly communicate or reveal to any person or persons any matter \nwhich shall be brought under my consideration or shall become known \nto me as a 

In [42]:
fourth_schedule = footer_remover(data[303:306])
print(fourth_schedule)

273
1[FOURTH SCHEDULE
[Articles 4(1) and 80(2)] 
Allocation of seats in the Council of States
To each State or Union territory specified in the first column of  the 
following table, there shall be allotted the number of seats specified in the 
second column thereof opposite to that State or that Union territory, as the case 
may be: 
TABLE
 1.
Andhra Pradesh ...................................................
2[11]
3[2.
Telangana ...........................................................
7]
4[3.]
Assam .................................................................
7
4[4.]
Bihar ...................................................................
5[16]
6[4[5.]
Jharkhand ............................................................
6]
7[8[4[6.]
Goa .....................................................................
1]]
9[8[4[7.]
Gujarat ................................................................
11]]
10[8[4[8.]
Haryana ...........................................................

In [43]:
fifth_schedule = footer_remover(data[306:310])
sixth_schedule = footer_remover(data[310:340])
seventh_schedule = footer_remover(data[340:355])
eighth_schedule = footer_remover(data[355:357])
ninth_schedule = footer_remover(data[357:375])
tenth_schedule = footer_remover(data[375:379])
eleventh_schedule =footer_remover(data[379:380])
twelfth_schedule = footer_remover(data[380:381])

In [44]:
schedules = []
schedules.append(first_schedule)
schedules.append(second_schedule)
schedules.append(third_schedule)
schedules.append(fourth_schedule)
schedules.append(fifth_schedule)
schedules.append(sixth_schedule)
schedules.append(seventh_schedule)
schedules.append(eighth_schedule)
schedules.append(ninth_schedule)
schedules.append(tenth_schedule)
schedules.append(eleventh_schedule)
schedules.append(twelfth_schedule)

In [45]:
print(schedules[0])

253
 
1[FIRST SCHEDULE 
[Articles 1 and 4]
I. THE STATES 
 
Name
Territories
1. 
Andhra 
Pradesh
 
 
2[The territories specified in sub-section (1) of section 3 of 
the Andhra State Act, 1953, sub-section (1) of section 3 of  
the States Reorganisation Act, 1956, the First Schedule to 
the Andhra Pradesh and Madras (Alteration of Boundaries) 
Act, 1959, and the Schedule to the Andhra Pradesh and 
Mysore (Transfer of Territory) Act, 1968, but excluding 
the territories specified in the Second Schedule to the 
Andhra Pradesh and Madras (Alteration of Boundaries) 
Act, 1959] 3[and the territories specified in section 3 of 
the Andhra Pradesh Reorganisation Act, 2014].
2. Assam
The 
territories 
which 
immediately 
before 
the 
commencement of this Constitution were comprised in the 
Province of Assam, the Khasi States and the Assam Tribal 
Areas, 
but 
excluding 
the 
territories 
specified 
in 
the 
Schedule 
to 
the 
Assam 
(Alteration of Boundaries) Act, 1951 4[and the territories 
spe

In [46]:
schedule_names = [
    "schedule 1",
    "schedule 2",
    "schedule 3",
    "schedule 4",
    "schedule 5",
    "schedule 6",
    "schedule 7",
    "schedule 8",
    "schedule 9",
    "schedule 10",
    "schedule 11",
    "schedule 12"
]

In [47]:
from langchain_text_splitters import CharacterTextSplitter
chucked_list = []
t_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=300,
    length_function=len,
    is_separator_regex=True,
)

for i,chapter in enumerate(schedules):
  # adding meta data here li is the list with name of chapters which we created above
  metadatas = [{"source": "https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf",
                "source_name": "Indian Constitution", "country":"India","db_owner":"LawGlance","schedule":schedule_names[i],}]
  # Use the splitter to split each chapter into chunks
  chunks = t_splitter.create_documents([chapter],metadatas=metadatas)
  # Append the chunks to the chunked_list
  chunked_list.append(chunks)


In [48]:
chunked_list[0]

[Document(metadata={'source': 'https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf', 'source_name': 'Indian Constitution', 'country': 'India', 'db_owner': 'LawGlance', 'part': 'PART I', 'part_name': 'The Union And Its Territories'}, page_content='PART I\nTHE UNION AND ITS TERRITORY Article 1. Name and territory of the Union.—(1) India, that is Bharat, \nshall be a Union of States.\n1[(2) The States and the territories thereof shall be as specified in \nthe First Schedule.]\n(3) The territory of India shall comprise—\n(a) the territories of the States; \n2[(b) the Union territories specified in the First Schedule; \nand]\n(c) such other territories as may be acquired. Article 2. Admission or establishment of new States.—Parliament may \nby law admit into the Union, or establish, new States on such terms and \nconditions as it thinks fit. Article 2A. [Sikkim to be associated with the Union.] —Omitted by the \nConstitution (Thirty-sixth A

In [49]:
def flatten_extend(matrix):
  flat_list = []
  for row in matrix:
    flat_list.extend(row)
  return flat_list
do = flatten_extend(chunked_list)
persistant_directory = "chroma_db_legal_bot_part1"
db1 = Chroma(persist_directory=persistant_directory, embedding_function=embeddings)
db1.add_documents(documents=do)

['23968ea0-288f-4387-b822-6cf34589a5ab',
 'aab777c8-b71d-4302-ab0e-68eac7ab3f96',
 '939e4739-e91f-44a2-8f50-bf9c0ecb0ec7',
 'beb7c453-e2c8-48dd-bc3b-ee3af03f1b39',
 'd8bd10e2-818f-4c84-b2ba-565c6b10a9bb',
 'a5bd2fa1-4940-4b4c-ab89-824030a3795b',
 'c6902bf9-f438-4089-b8d4-ba2fc6afcf1a',
 'eedbfd8f-c33b-46aa-b600-2bf254510b33',
 '588a19ba-81f0-469e-b25c-a6288e4e3838',
 'e8678142-9232-467a-82fc-5f33eb90be5a',
 '2c57270c-9e98-4d84-8231-38a5a2f14c55',
 '2621e708-9b83-4564-a1c3-da3647cf0ef9',
 '94087f8f-cc41-42d7-a3c6-a61933d97d24',
 'b3041766-19fe-490b-9934-b364bfa4abc4',
 'b2ce6de0-2205-4b87-b578-7578aaf66180',
 '0bc92742-ac40-41a9-bb08-63262fd34a7c',
 'db591d64-e22c-44cc-b66b-86591ec8162e',
 '031eeebf-dd4e-4c11-83e2-2b4618cc4280',
 '0bcde914-f67a-4aa8-a393-6e401958c797',
 'a1d11d2a-8b33-4fec-8203-b607e0230a9b',
 '11ee9d62-4596-4171-a01b-af63ccba306c',
 '7200f49c-d1dd-43e4-8b81-270a2087713d',
 '2d5333ff-d315-4ddc-8764-01dc968cbb31',
 '016eacf0-af3d-4c29-80e8-5a6827612fd8',
 '1fda58dd-3bf8-

In [50]:
pr = data[3]
pr.metadata.clear()
pr

Document(metadata={}, page_content='THE CONSTITUTION OF INDIA\n____________                                                                    \nCONTENTS\n__________\n                                                                                          \nPREAMBLE\nPART I \nTHE UNION AND ITS TERRITORY\nARTICLES \n1.\nName and territory of the Union.\n  2. \nAdmission or establishment of new States. \n[2A.         Sikkim to be associated with the Union.—Omitted.]\n3.\nFormation of new States and alteration of areas, boundaries or \nnames of existing States.\n4.\nLaws made under articles 2 and 3 to provide for the amendment of \nthe First and the Fourth Schedules and supplemental, incidental \nand consequential matters.\nPART II\nCITIZENSHIP\n5.\nCitizenship at the commencement of the Constitution.\n6.\nRights of citizenship of certain persons who have migrated to \nIndia from Pakistan.\n7.\nRights of citizenship of certain migrants to Pakistan.\n8.\nRights of citizenship of certain p

In [51]:
# Delete all old metadata
# pr.metadata.clear()

# Add new metadata
pr.metadata = {"source": "https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf",
                "source_name": "Indian Constitution", "country":"India","db_owner":"LawGlance","preamble": "preamble",}

# Now the old metadata is completely removed and replaced with the new one


In [52]:
pr

Document(metadata={'source': 'https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf', 'source_name': 'Indian Constitution', 'country': 'India', 'db_owner': 'LawGlance', 'preamble': 'preamble'}, page_content='THE CONSTITUTION OF INDIA\n____________                                                                    \nCONTENTS\n__________\n                                                                                          \nPREAMBLE\nPART I \nTHE UNION AND ITS TERRITORY\nARTICLES \n1.\nName and territory of the Union.\n  2. \nAdmission or establishment of new States. \n[2A.         Sikkim to be associated with the Union.—Omitted.]\n3.\nFormation of new States and alteration of areas, boundaries or \nnames of existing States.\n4.\nLaws made under articles 2 and 3 to provide for the amendment of \nthe First and the Fourth Schedules and supplemental, incidental \nand consequential matters.\nPART II\nCITIZENSHIP\n5.\nCitizenship at the co

In [53]:
db1.add_documents(documents=[pr])

['58bdce83-ac65-4cdf-a2b0-5155ac129dd5']

In [54]:
from langchain_core.documents import Document

In [55]:
greetings = [
    {"text": "Hello", "language": "English"},
    {"text": "Hi", "language": "English"},
    {"text": "Hey", "language": "English"},
    {"text": "Greetings", "language": "English"},
    {"text": "Good Morning", "language": "English"},
    {"text": "Good Afternoon", "language": "English"},
    {"text": "Good Evening", "language": "English"},
    {"text": "Howdy", "language": "English"},
    {"text": "What's up?", "language": "English"},
    {"text": "Welcome", "language": "English"},
    {"text": "Salutations", "language": "English"},
    {"text": "Hiya", "language": "English"},
    {"text": "Yo", "language": "English"},
    {"text": "How’s it going?", "language": "English"},
    {"text": "Nice to meet you", "language": "English"},
    {"text": "How are you?", "language": "English"},
    {"text": "Good day", "language": "English"},
    {"text": "Pleased to meet you", "language": "English"},
    {"text": "Hey there", "language": "English"},
    {"text": "What's new?", "language": "English"},
    {"text": "Long time no see", "language": "English"},
    {"text": "How have you been?", "language": "English"},
    {"text": "How do you do?", "language": "English"},
    {"text": "Good to see you", "language": "English"},
    {"text": "How are things?", "language": "English"},
    {"text": "Hello there", "language": "English"},
    {"text": "Hi there", "language": "English"},
]

# Convert the greetings into a list of LangChain Documents with language metadata
documents = [Document(page_content=greeting["text"], metadata={"source": "english greeting words","db_owner":"LawGlance","language": greeting["language"]})
    for greeting in greetings
]


### Using CrewAI in the legal domain

Thie notebook utilises an exemplary application of Crew AI in the legal domain, showcasing the power of multi-agent workflows to streamline complex legal research tasks. It combines Crew AI, LangChain, and Chroma to retrieve legal documents, perform web searches, and deliver concise, accurate answers tailored to user queries.

This notebook serves as a practical demonstration of how Crew AI can be applied in the legal field, making it a valuable resource for legal professionals, researchers, and developers exploring AI-driven solutions. Future enhancements include multilingual support, voice interaction, and mobile deployment, demonstrating its scalability and versatility.

### Agentic RAG Using CrewAI & LangChain for Legaldomain

In [88]:
# importing necessary libraries
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool # A decorator used to turn any Python function into a "Tool" that an AI agent can understand and use (like a search function or a calculator).

# CrewAI Architecture: The Digital Law Firm

To handle complex legal queries from the Indian Constitution and BNS, we use a **Multi-Agent Orchestration** approach rather than a single chatbot. This prevents "context overload" and ensures higher accuracy.

### Core Components

| Component | Real-World Analogy | Technical Purpose |
| :--- | :--- | :--- |
| **Agent** | **The Specialist** | An autonomous unit with a specific **Role** (e.g., Legal Researcher), **Goal**, and **Backstory**. It chooses which tools to use. |
| **Task** | **The Assignment** | A specific to-do item (e.g., "Extract Article 21 details"). It defines the requirements and assigns them to an Agent. |
| **Tool** | **The Equipment** | External functions (Skills) given to Agents, such as `ChromaRetriever` or `InternetSearch`. |
| **Crew** | **The Team/Manager** | The logic that brings everyone together. It manages the **Process** (Sequential or Hierarchical) and final output. |



### Why use a "Crew" for LawRAG?

A single LLM can get confused by 400+ pages of legal text. By splitting the logic into a Crew, we achieve better results:

1.  **Specialization:** * **Agent 1 (Researcher):** Focuses solely on finding the exact legal clauses in the `Chroma` database.
    * **Agent 2 (Analyst):** Takes those raw clauses and simplifies them for a common citizen.
2.  **Accuracy:** By giving the Researcher a specific `Chroma` tool, we minimize "hallucinations" because the agent is forced to look at the actual PDF data first.
3.  **Scalability:** We can easily add a "Bilingual Agent" or a "Case Law Expert" later without rewriting the entire system.

In [89]:
import getpass
"""
This module provides a way to type in sensitive information (like passwords or API keys) securely. When you type, 
the characters remain hidden—exactly like when you type a password into a terminal.
"""
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("GROQ_API_KEY")
_set_env('TAVILY_API_KEY')

### 🛡️ Secure Environment Configuration

This block ensures that sensitive API keys are handled securely without being hardcoded into the notebook. This is critical for preventing security leaks when sharing code or pushing to GitHub.

#### 📝 Code Breakdown

1.  **`import getpass`**: Imports a utility that hides user input (like a password field) so keys aren't visible on the screen while typing.
2.  **`import os`**: Used to set and retrieve "Environment Variables" that the AI libraries (LangChain, CrewAI) look for automatically.
3.  **The `_set_env` Function**:
    * **Check:** It first checks if the variable already exists (`os.environ.get(var)`).
    * **Prompt:** If the key is missing, it opens a secure input box at the top of the screen.
    * **Assignment:** It saves the input into the session's memory so it can be accessed globally by the code.

#### ⚙️ Variables Being Set
* **`OPENAI_API_KEY`**: Powers the "Brain" of the agents (GPT models).
* **`TAVILY_API_KEY`**: Powers the "Internet Search" capability for real-time legal updates.



> **Tip:** If you are using **Google Colab**, you can also use the **"Secrets"** (🔑 icon) on the left sidebar to store these keys permanently. This code will automatically skip the prompt if you've already set them there.

### 🌐 Tavily API: The "Search Engine" for AI

While standard search engines (like Google) are designed for humans to browse websites, **Tavily** is a search engine built specifically for **AI Agents**.

#### 🔑 What is the TAVILY_API_KEY?
The API Key is your unique identifier that allows your Agent to "talk" to the internet. It acts as a bridge between your local code and the live web.

#### 🚀 Why use it in LawGlance?
1. **Live Updates:** Legal databases (like our Chroma DB) have a "cutoff date." Tavily allows the agent to find the most recent amendments or news from 2025-2026.
2. **AI-Ready Results:** Unlike Google, which returns messy HTML and ads, Tavily returns **clean text** that the AI can read instantly without getting confused.
3. **Fact-Checking:** It allows the agent to verify internal knowledge against official government websites in real-time.
4. **Token Saving:** Because Tavily removes "junk" text (like navigation menus and ads) before sending data to the AI, it saves you money on your OpenAI/Groq token usage.

#### 🛠️ How it works in the CrewAI Workflow
* **The Problem:** A user asks about a very new law not in the PDF.
* **The Decision:** The Agent realizes it doesn't have the answer in its "memory" (Chroma DB).
* **The Action:** Using the `TAVILY_API_KEY`, the agent searches the web, reads the top 5 relevant sites, and synthesizes a factual answer.



> **Note:** Tavily offers a **Free Tier** (1,000 searches/month), which is perfect for development and testing.

#### **Use the Native CrewAI LLM Class**
Instead of using ChatGroq from LangChain, use the native LLM class provided by CrewAI. This is the most stable way to use Groq with the current version of CrewAI.

In [90]:
from crewai import LLM

# Define your Groq LLM correctly
llm = LLM(
    model="groq/llama-3.1-8b-instant", # or your specific model
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1", # Crucial for native support
    temperature=0.7
)

In [91]:
vector_store = Chroma(
    persist_directory="chroma_db_legal_bot_part1",
    embedding_function=embeddings
)

retriever = vector_store.as_retriever(
    search_type = "similarity_score_threshold", 
    search_kwargs={
        "k": 2, 
        "score_threshold":0.5
        }
)

### We are creating a Custom tool for retrieval

In [92]:
@tool("ChromaRetriever")
def chroma_retriever_tool(query : str):
    # Assuming 'retriever' is your Chroma retriever instance
    """Retrieves relevant documents from the Chroma vector store."""
    results = retriever.invoke(query)
    return results

chroma_tool = chroma_retriever_tool

### Custom tool for web search
We are using Tavily web search

In [101]:
from langchain_community.tools import TavilySearchResults
from crewai.tools import BaseTool
from pydantic import Field

search = TavilySearchResults()

class SearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Useful for search-based queries. Use this to find current information about latest legal trends and news."
    search: TavilySearchResults = Field(default_factory=TavilySearchResults)

    def _run(self, query: str) -> str:
        """Execute the search query and return results"""
        try:
            return self.search.run(query)
        except Exception as e:
            return f"Error performing search: {str(e)}"
web_search_tool = SearchTool()

### 🛠️ Custom Tooling: Class-Based Search Tool

In CrewAI, while simple tools can use the `@tool` decorator, **Class-Based Tools** (inheriting from `BaseTool`) provide better error handling, clearer structure, and stricter data validation using Pydantic.

#### 📝 Code Logic & Structure

1.  **`BaseTool` Inheritance**: Makes the tool compatible with the CrewAI framework, allowing agents to understand its name, description, and usage rules.
2.  **The "Agent's Manual" (`name` & `description`)**:
    * **Name**: The label in the agent's toolbox.
    * **Description**: This is the most critical part. The LLM reads this to decide *when* to use the tool. In this case, it knows to use it for "latest legal trends and news."
3.  **Pydantic `Field`**: Ensures the `TavilySearchResults` engine is initialized correctly every time the tool is called.
4.  **The `_run` Method**: The internal logic of the tool. It includes a `try-except` block to ensure that if the search fails (e.g., API timeout), the Agent receives a text error instead of the entire code crashing.

#### 🚀 Implementation Pattern

```python
from crewai.tools import BaseTool
from pydantic import Field
from langchain_community.tools import TavilySearchResults

class SearchTool(BaseTool):
    name: str = "Search"
    description: str = "Useful for finding current information about latest legal trends and news."
    search: TavilySearchResults = Field(default_factory=TavilySearchResults)

    def _run(self, query: str) -> str:
        try:
            return self.search.run(query)
        except Exception as e:
            return f"Error performing search: {str(e)}"

# Initialize the tool
web_search_tool = SearchTool()

### 🛠️ Class-Based Tooling (CrewAI)
The `BaseTool` approach is superior for production because:
1. **Self-Contained:** The `search` logic and the `tool` metadata stay in one object.
2. **Resilient:** The `_run` method handles errors internally, preventing agent crashes.
3. **Descriptive:** It uses Pydantic to clearly define what the tool needs (input) and what it does (description).

### Agents

In [102]:
retriever_agent = Agent(
    role='Retriever Agent',
    goal='Retrieve relevant content about "{query}" from the vector store.',
    backstory=("You are skilled at searching the vector store for user queries and fetching relevant documents."
               "Your ability to find and retrieve relevant content ensures accurate reports."),
    verbose=True,
    memory = True,
    tools=[chroma_tool],
    llm=llm
)

legal_assistant_agent = Agent(
    role="Legal Assistant Agent",
    goal="Generate responses for the {query} based on retrieved documents only",
    backstory=
    "You are a lawyer assistant LawGlance and you answer for legal related queries"
    "You create informative responses using the data provided by the retriever task only"
    "If an informative response can't be provided from the documents you should use websearch tool and respond based on it."
    "Only use the tools explicitly provided to you: 'web_search' and 'ChromaRetriever'. Do not attempt to use any other search engines.",
    verbose=True,
    memory = True,
    allow_delegation=False,
    tools = [web_search_tool],
    llm=llm
)

evaluation_agent = Agent(
    role="Evaluation Expert Agent",
    goal="Verify and evaluate the accuracy and authenticity of responses created by retriever and generator agents.",
    backstory=
    "You are an evaluation expert in the LawGlance ecosystem. "
    "Your primary task is to validate the responses generated by the retriever and generator agents. "
    "You check for accuracy, relevance, and authenticity of the content before it reaches the customer.",
    verbose=True,
    memory=False,
    allow_delegation=False,
    tools = [web_search_tool],
    llm=llm
)

editor_agent = Agent(
    role="Editor Agent",
    goal="Create a concise and edited output for '{query}' based on the generated response.",
    backstory=(
        "You are an Editor tasked with refining the generated responses."
        "You ensure that the final output is concise,to the point, relevant, and properly formatted without any hallucinations."
        "For responses generated using web search, you must include the source of the information."
    ),
    verbose=True,
    memory=False,
    llm=llm
)

### Tasks

In [97]:
retrieval_task = Task(
    description="Retrieve documents related to '{query}' in order to search the vector store."
    "Reframe a standalone question based on the memory.If a standalone question can't be created search the query as it is.",
    expected_output="List of relevant documents.",
    agent = retriever_agent,
)

generation_task = Task(
    description="Generate a response for the {query} based on the retrieved documents only.",
    expected_output="A proper response to the user's query based on the relevant documents only.",
    agent=legal_assistant_agent,
    context=[retrieval_task]
)

evaluation_task = Task(
    description="Verify the accuracy and authenticity of the response generated for the '{query}' based on retrieved documents and validate on basis of stand alone websearch based on response generated.",
    expected_output="A verified and accurate response ready to be shared with the customer, or an error indicating issues with the response and provide with percentage of accuracy",
    agent=evaluation_agent,
    context=[generation_task]
)

editing_task = Task(
    description=(
        "Edit the response generated for '{query}' to ensure conciseness (2-3 sentences), "
        "proper formatting, and source attribution if the response includes web search-based content."
    ),
    expected_output="A concise and edited response that aligns with the query and includes sources where applicable.Ensure it is limited to 2-3 sentences and includes the source for web search-based answers.",
    agent=editor_agent,
    context=[generation_task]
)

### 📋 Task Orchestration (The LawGlance Pipeline)

Tasks in CrewAI define the specific execution steps and the data flow between agents.

1. **Retrieval Task**: Converts the user's query into a standalone search query and fetches documents from the local Vector Database.
2. **Generation Task**: Drafts a legal response. It uses the `context` parameter to ensure it only uses the documents found in the previous step.
3. **Evaluation Task**: A "Double-Check" step that verifies the drafted answer against the web to calculate an accuracy score.
4. **Editing Task**: The final cleanup. It enforces constraints like "2-3 sentences" and "source attribution."

#### 🔗 The Power of Context
By using `context=[task_name]`, we create a **Sequential Chain of Thought**. This ensures that the AI's final answer is "grounded" in your legal PDFs rather than its general training data.

### The Crew

In [107]:
# Creatig a crew with the agents and tasks
my_crew = Crew(
    agents=[retriever_agent,legal_assistant_agent,evaluation_agent,editor_agent], 
    tasks=[retrieval_task,generation_task,evaluation_task,editing_task],
    verbose=True,
    process=Process.sequential,
    max_rpm=2, # <--- Add this! It limits the crew to 2 requests per minute.,
    cache=True # Helps prevent re-calling tools if the agent loops
    )

### 🚢 The Crew: Orchestrating the Workflow

The `Crew` object is the final container that brings our **Agents** and **Tasks** together into a collaborative system.

* **Collaboration:** It manages the data hand-off between agents (e.g., passing retrieved documents to the writer).
* **Process Flow:** By default, it follows a **Sequential Process**, executing tasks in the exact order they are listed in the `tasks` array.
* **Execution:** We use the `.kickoff()` method to trigger the entire pipeline, passing in the user's `{query}` as an input variable.

> **Key Note:** Even though we have 4 agents, the user only interacts with the **Crew**. The internal "meetings" and fact-checking between agents happen automatically behind the scenes.

### 🚀 Executing the LawRAG Crew

This is the final step where we provide a real-world question and trigger the multi-agent pipeline.

* **`kickoff`**: Starts the sequential process where each agent performs its task in order.
* **`inputs`**: Pass dynamic variables into the system. The key `"query"` must match the `{query}` placeholders used in your Agent goals and Task descriptions.
* **`results.raw`**: Returns the final string output from the last task (the Editing Task). 

#### 📊 Execution Stats:
After running this, CrewAI will also provide metadata in the background, such as:
* **Token Usage**: How many tokens each agent consumed.
* **Execution Time**: How long the total research took.

In [ ]:
user_query = "What is relevant laws and articles dealing with rights of women in Indian Constitution"
results = my_crew.kickoff(inputs={"query": user_query})
print(f"Raw Output: {results.raw}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  21e66f7f-eba8-467b-ad50-76e12139d010                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Retrieve documents related to 'What is relevant laws and articles dealing with rights of women in        │
│  Indian Constitution' in order to search the vector store.Reframe a standalone question based on the memory.If  │
│  a standalone question can't be created search the query as it is.                                              │
│  ID: 4031388e-20c1-47c0-852c-28826d0ed6f8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever Agent                                                                                         │
│                                                                                                                 │
│  Task: Retrieve documents related to 'What is relevant laws and articles dealing with rights of women in        │
│  Indian Constitution' in order to search the vector store.Reframe a standalone question based on the memory.If  │
│  a standalone question can't be created search the query as it is.                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: chroma_retriever                                                                                         │
│  Args: {'query': 'What are relevant laws and articles dealing with rights of women in Indian Constitution'}     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool chroma_retriever executed with result: [Document(id='a4e89012-1195-47e7-a2a2-389e88002b4d', metadata={'country': 'India', 'source': 'https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf', '...


...

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  21e66f7f-eba8-467b-ad50-76e12139d010                                                                           │
│  Final Output: **Relevant Laws and Articles Dealing with Rights of Women in Indian Constitution**               │
│                                                                                                                 │
│  The Indian Constitution ensures equal rights and protection for women through the following articles: Article  │
│  14, which guarantees equality before the law, and Article 15, which prohibits discrimination against any       │
│  citizen, including women, on grounds of sex, religion, caste, or place of birth (Source: Government of         │
│  India). These articles aim to promote equality and prevent discrimination against women in all aspects of      │
│  life. They are fundamental to upholding the rights of women in India.                                          │
│                                                                                                                 │
│  **Source:**                                                                                                    │
│  - Government of India. (n.d.). Indian Constitution.                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Raw Output: **Relevant Laws and Articles Dealing with Rights of Women in Indian Constitution**

The Indian Constitution ensures equal rights and protection for women through the following articles: Article 14, which guarantees equality before the law, and Article 15, which prohibits discrimination against any citizen, including women, on grounds of sex, religion, caste, or place of birth (Source: Government of India). These articles aim to promote equality and prevent discrimination against women in all aspects of life. They are fundamental to upholding the rights of women in India.

**Source:**
- Government of India. (n.d.). Indian Constitution.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



┌───────────────────────────── Execution Traces ──────────────────────────────┐
│                                                                             │
│  🔍 Detailed execution traces are available!                                │
│                                                                             │
│  View insights including:                                                   │
│    • Agent decision-making process                                          │
│    • Task execution flow and timing                                         │
│    • Tool usage details                                                     │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
Would you like to view your execution traces? [y/N] (20s timeout): 



┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                                             │
│  Info: Tracing has been disabled.                                           │
│                                                                             │
│  Your preference has been saved. Future Crew/Flow executions will not       │
│  collect traces.                                                            │
│                                                                             │
│  To enable tracing later, do any one of these:                              │
│  • Set tracing=True in your Crew/Flow code                                  │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file              │
│  • Run: crewai traces enable                                                │
│                                                                             │
└─────────────────────────────────────

In [109]:

print(results.raw)

**Relevant Laws and Articles Dealing with Rights of Women in Indian Constitution**

The Indian Constitution ensures equal rights and protection for women through the following articles: Article 14, which guarantees equality before the law, and Article 15, which prohibits discrimination against any citizen, including women, on grounds of sex, religion, caste, or place of birth (Source: Government of India). These articles aim to promote equality and prevent discrimination against women in all aspects of life. They are fundamental to upholding the rights of women in India.

**Source:**
- Government of India. (n.d.). Indian Constitution.


In [110]:
import json
print(f"Raw Output: {results.raw}")
if results.json_dict:
    print(f"JSON Output: {json.dumps(results.json_dict, indent=2)}")
if results.pydantic:
    print(f"Pydantic Output: {results.pydantic}")
print(f"Tasks Output: {results.tasks_output}")
print(f"Token Usage: {results.token_usage}")

Raw Output: **Relevant Laws and Articles Dealing with Rights of Women in Indian Constitution**

The Indian Constitution ensures equal rights and protection for women through the following articles: Article 14, which guarantees equality before the law, and Article 15, which prohibits discrimination against any citizen, including women, on grounds of sex, religion, caste, or place of birth (Source: Government of India). These articles aim to promote equality and prevent discrimination against women in all aspects of life. They are fundamental to upholding the rights of women in India.

**Source:**
- Government of India. (n.d.). Indian Constitution.
Tasks Output: [TaskOutput(description="Retrieve documents related to 'What is relevant laws and articles dealing with rights of women in Indian Constitution' in order to search the vector store.Reframe a standalone question based on the memory.If a standalone question can't be created search the query as it is.", name="Retrieve documents relat